# 第四阶段：STL（标准模板库）

## 实验 8：从类型读懂 SDK API Contract

前七个实验分别学习了拥有型容器、非拥有视图和显式状态类型。本实验不再引入新的 STL 类型，而是把它们组合成接近真实 Native SDK Core 的接口。

目标不是记住“span 是一个模板类”，而是看到函数签名就回答：

- 谁拥有输入和输出？
- 哪个参数只是临时借用？
- 数据能否修改、是否连续？
- 返回值是否可能缺失或失败？
- 哪些 C++ 语义必须在 C ABI 边界重新编码？

核心原则是：**类型本身就是 ownership 和 API contract。**

In [ ]:
// 本步骤：引入连续容器、视图、状态类型、算法和 ABI 示例所需的标准库。
#include <algorithm>
#include <array>
#include <cassert>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <numeric>
#include <optional>
#include <span>
#include <string>
#include <string_view>
#include <utility>
#include <variant>
#include <vector>

### 1. 先建立类型词汇表

| 类型 | Ownership | Contract |
| --- | --- | --- |
| `std::string` | Own | 拥有可变长度文本 |
| `std::string_view` | Borrow | 借用连续只读字符 |
| `std::vector<T>` | Own | 拥有动态长度连续元素 |
| `std::array<T, N>` | Own | 拥有编译期固定数量元素 |
| `std::span<T>` | Borrow | 借用可写连续元素 |
| `std::span<const T>` | Borrow | 借用只读连续元素 |
| `std::optional<T>` | Own state/value | 包含一个 `T` 或正常缺失 |
| `std::variant<A, B>` | Own state/value | 当前包含 A 或 B |

optional/variant 拥有 contained value，但若 `T` 本身是 pointer、reference wrapper 或 view，它们拥有的仍只是借用对象。

### 2. `vector` 返回值表达拥有的数据

看到：

```cpp
std::vector<std::uint8_t> read_file();
```

应直接翻译为：函数产生动态字节数据，并按值把 owner 交给调用方。调用结束后，调用方可以保存或修改结果；函数内部不再拥有返回的 vector。

下面用固定字节模拟已经从文件读取的内容，只研究返回类型的所有权语义。

In [ ]:
// 本步骤：定义返回 owning vector 的文件读取接口。
std::vector<std::uint8_t> read_file()
{
    // 局部 vector 拥有模拟文件内容，返回时所有权随值结果交给调用方。
    std::vector<std::uint8_t> content{
        0x4b,
        0x4e,
        0x01,
        0x02};

    return content;
}

In [ ]:
// 本步骤：接收 read_file 的拥有型结果，并在函数返回后继续使用。
{
    std::vector<std::uint8_t> bytes = read_file();

    // 调用方现在拥有 vector，可以安全保存和修改它。
    bytes.push_back(0x03);

    assert(bytes.size() == 5);
    assert(bytes.front() == 0x4b);

    std::cout << "owned bytes = " << bytes.size() << '\n';
}

### 3. `span<const T>` 参数表达只读 buffer 借用

看到：

```cpp
void process(std::span<const std::uint8_t> input);
```

应直接翻译为：函数在调用期间借用一段连续字节，不能通过该 view 修改元素，也不能在调用结束后保存 span。调用方仍拥有底层 buffer。

span 按值传递只复制轻量视图，不复制字节。

In [ ]:
// 本步骤：定义只依赖同步只读借用的 buffer 处理函数。
void inspect_input(
    std::span<const std::uint8_t> input)
{
    // 只在 span 边界内读取元素，不保存 input 或 data()。
    const std::uint64_t checksum =
        std::accumulate(
            input.begin(),
            input.end(),
            std::uint64_t{0});

    std::cout << "input checksum = " << checksum << '\n';
}

In [ ]:
// 本步骤：从 array 和 vector 借出只读 span，验证 owner 不会转移。
{
    std::array<std::uint8_t, 3> fixed{1, 2, 3};
    std::vector<std::uint8_t> dynamic{4, 5, 6};

    // 两次借用都只覆盖函数调用，callee 不拥有任何 buffer。
    inspect_input(fixed);
    inspect_input(dynamic);

    // 调用返回后，原容器仍由调用方拥有并保持可用。
    assert(fixed[0] == 1);
    assert(dynamic[0] == 4);
}

### 4. `string_view + optional<User>` 同时表达借用和缺失

看到：

```cpp
std::optional<User> find_user(std::string_view name);
```

应翻译为两层语义：

1. `name` 是调用期间的只读字符串借用，函数不能让 view 逃逸；
2. 返回值拥有一个 `User`，或者表示用户正常不存在。

这里的返回值不是指向内部数据库对象的裸指针。找到用户时，调用方获得独立的 User 值。

In [ ]:
// 本步骤：定义拥有名称的 User，以及可能找不到用户的查询接口。
struct User
{
    std::int32_t id;
    std::string name;
};

std::optional<User> find_user(std::string_view name)
{
    // 只在本次调用内比较借用字符，不保存 string_view。
    if (name == "Bob")
    {
        return User{1, "Bob"};
    }

    if (name == "Alice")
    {
        return User{2, "Alice"};
    }

    // 未找到是正常状态，不使用 magic id 或悬空 pointer。
    return std::nullopt;
}

In [ ]:
// 本步骤：查询存在和不存在的用户，并分别处理 optional 状态。
{
    std::string query = "Bob";

    // query 覆盖完整调用；返回的 User 拥有自己的 name。
    const std::optional<User> found = find_user(query);
    assert(found.has_value());
    assert(found->id == 1);
    assert(found->name == "Bob");

    const std::optional<User> missing =
        find_user("Unknown");

    // 空 optional 明确表示正常未找到。
    assert(!missing);
}

### 5. `variant<Result, Error>` 返回多个明确状态

看到：

```cpp
std::variant<Result, Error> execute();
```

应直接翻译为：调用返回一个拥有型结果对象，active alternative 要么是 Result，要么是 Error。调用方必须检查或访问正确状态；不存在额外的 null 状态。

与 `optional<Result>` 相比，Error alternative 能保留失败原因。

In [ ]:
// 本步骤：定义成功、错误及其封闭结果类型。
struct Result
{
    std::int32_t processed_count;
};

enum class ErrorCode
{
    empty_input,
    execution_failed
};

struct Error
{
    ErrorCode code;
    std::string message;
};

using ExecuteOutcome =
    std::variant<Result, Error>;

ExecuteOutcome execute()
{
    // 本示例返回成功状态；Result 由 variant 按值拥有。
    return Result{4};
}

ExecuteOutcome execute_failure_example()
{
    // 独立 helper 用于观察 Error alternative。
    return Error{
        ErrorCode::execution_failed,
        "execution failed"};
}

In [ ]:
// 本步骤：分别访问 execute 的成功状态和示例失败状态。
{
    const ExecuteOutcome success = execute();

    // get_if 返回 contained Result 的临时借用。
    const Result *result =
        std::get_if<Result>(&success);
    assert(result != nullptr);
    assert(result->processed_count == 4);

    const ExecuteOutcome failure =
        execute_failure_example();

    // Error 是另一合法 alternative，而不是 magic result。
    const Error *error =
        std::get_if<Error>(&failure);
    assert(error != nullptr);
    assert(error->code == ErrorCode::execution_failed);
}

### 6. 把 ownership 与状态组合成内部 Core API

现在组合出最终接口：

```cpp
std::variant<
    std::vector<std::uint8_t>,
    Error
>
process(
    std::span<const std::uint8_t> input
);
```

从类型可以直接读出：

```text
input
└─ span<const uint8_t>
   └─ 借用、只读、连续、仅在调用期间有效

return
└─ variant<vector<uint8_t>, Error>
   ├─ success: 调用方拥有动态输出 buffer
   └─ failure: 调用方拥有错误对象
```

In [ ]:
// 本步骤：实现借用只读输入、返回拥有型成功值或错误值的 Core。
using ProcessOutcome =
    std::variant<
        std::vector<std::uint8_t>,
        Error>;

class Processor
{
public:
    ProcessOutcome process(
        std::span<const std::uint8_t> input) const
    {
        // 空输入映射为拥有型 Error alternative。
        if (input.empty())
        {
            return Error{
                ErrorCode::empty_input,
                "input is empty"};
        }

        std::vector<std::uint8_t> output;

        // 一次预留完整结果空间，避免循环中重复分配。
        output.reserve(input.size());

        // 只读取 borrowed input，把新值写入 owning output。
        for (std::uint8_t value : input)
        {
            output.push_back(
                static_cast<std::uint8_t>(value + 1));
        }

        // 返回 vector 值，把结果 owner 交给调用方。
        return output;
    }
};

In [ ]:
// 本步骤：验证成功路径不修改输入，并把输出所有权交给调用方。
{
    const Processor processor;
    const std::array<std::uint8_t, 4> input{
        1,
        2,
        3,
        4};

    const ProcessOutcome outcome =
        processor.process(input);

    // 成功时借用 variant 内部拥有的 vector 进行检查。
    const std::vector<std::uint8_t> *output =
        std::get_if<std::vector<std::uint8_t>>(
            &outcome);

    assert(output != nullptr);
    assert(
        *output ==
        std::vector<std::uint8_t>({2, 3, 4, 5}));

    // const span 契约保证 Core 没有通过输入 view 修改元素。
    assert(
        (input ==
         std::array<std::uint8_t, 4>{1, 2, 3, 4}));
}

In [ ]:
// 本步骤：用空 span 触发错误 alternative，并读取明确错误原因。
{
    const Processor processor;
    const std::span<const std::uint8_t> empty;

    const ProcessOutcome outcome =
        processor.process(empty);

    // 空输入不返回空 vector，而是显式 Error 状态。
    const Error *error =
        std::get_if<Error>(&outcome);

    assert(error != nullptr);
    assert(error->code == ErrorCode::empty_input);
}

### 7. 为什么它适合作为 C++ 内部 API

这个签名适合 C++ Core，因为语言和标准库会共同维护：

- span 的 pointer + length 视图语义；
- `const` 元素约束；
- vector 的动态存储所有权和析构；
- variant 的 active alternative 与 contained object 生命周期；
- 按值返回、异常安全和 RAII。

调用方得到紧凑、类型安全且难以误用的接口。它无需手工配对 output pointer、capacity、实际长度和错误 tag。

### 8. 为什么绝对不能直接暴露为 C ABI

`std::span`、`std::vector`、`std::variant` 都是 C++ 类型，直接跨 ABI 会依赖：

- C++ name mangling 与调用约定；
- object layout、alignment 和 active tag 表示；
- constructor、destructor、copy/move 规则；
- template instantiation；
- standard library 实现；
- compiler、编译选项和 runtime compatibility；
- 异常展开与内存分配器边界。

C 编译器不理解这些类型及其生命周期。即使两侧都使用 C++，不同标准库或编译器版本也不能默认二进制兼容。因此 C ABI 必须重新表达 contract，而不是传递 STL 对象布局。

正确分层是：

```text
                 Internal C++

     span / vector / optional / variant
                      │
                      ▼
                   C++ Core

══════════════════ ABI Boundary ══════════════════

                     C ABI
                      │
                      ▼
 pointer + length / handle / fixed status / out parameter
                      │
                      ▼
          Kotlin/Native cinterop wrapper
                      │
                      ▼
       ByteArray / nullable / sealed result
```

C ABI 负责稳定和可互操作；C++ Core 与 Kotlin wrapper 各自在边界两侧恢复本语言的表达力。

### 9. 用 caller-owned output buffer 翻译 Core API

下面的 C ABI 采用两次调用模式：

1. 调用方先传空 output，函数通过 `output_size` 报告所需容量；
2. 调用方分配缓冲区后再次调用，Core 把结果复制到 caller-owned storage。

这样 C 边界只暴露 pointer、length、固定宽度 status 和 out parameter。对于大型或长期对象，也可以改用 opaque handle + destroy 函数。

In [ ]:
// 本步骤：定义固定宽度状态码，并把 C++ Core 结果翻译为 C 参数。
using sdk_status = std::int32_t;

inline constexpr sdk_status SDK_STATUS_OK = 0;
inline constexpr sdk_status SDK_STATUS_BUFFER_TOO_SMALL = 1;
inline constexpr sdk_status SDK_STATUS_EMPTY_INPUT = 2;
inline constexpr sdk_status SDK_STATUS_INVALID_ARGUMENT = 3;
inline constexpr sdk_status SDK_STATUS_INTERNAL_ERROR = 4;

extern "C" sdk_status sdk_process(
    const std::uint8_t *input,
    std::size_t input_size,
    std::uint8_t *output,
    std::size_t output_capacity,
    std::size_t *output_size) noexcept
{
    // 必需 out parameter 不能为空；非空输入必须有有效地址。
    if (output_size == nullptr ||
        (input == nullptr && input_size != 0))
    {
        return SDK_STATUS_INVALID_ARGUMENT;
    }

    // 在构造 span 前单独处理空输入，不让空指针进入 view。
    if (input_size == 0)
    {
        return SDK_STATUS_EMPTY_INPUT;
    }

    try
    {
        const Processor processor;

        // 只在同步调用内把 C pointer + length 包装成 span。
        const ProcessOutcome outcome =
            processor.process(
                std::span<const std::uint8_t>(
                    input,
                    input_size));

        if (const Error *error =
                std::get_if<Error>(&outcome))
        {
            // 把 C++ Error alternative 映射为固定公开状态。
            return error->code == ErrorCode::empty_input
                ? SDK_STATUS_EMPTY_INPUT
                : SDK_STATUS_INTERNAL_ERROR;
        }

        const std::vector<std::uint8_t> &result =
            std::get<std::vector<std::uint8_t>>(
                outcome);

        // 无论容量是否足够，都先报告准确的所需长度。
        *output_size = result.size();

        if (output == nullptr ||
            output_capacity < result.size())
        {
            return SDK_STATUS_BUFFER_TOO_SMALL;
        }

        // 容量验证完成后，把 Core-owned 临时结果复制给调用方。
        std::copy(
            result.begin(),
            result.end(),
            output);

        return SDK_STATUS_OK;
    }
    catch (...)
    {
        // 所有 C++ 异常都在 ABI 内部转换，绝不穿过 C 边界。
        return SDK_STATUS_INTERNAL_ERROR;
    }
}

In [ ]:
// 本步骤：按查询容量、分配输出、再次调用的顺序使用 C ABI。
{
    const std::array<std::uint8_t, 4> input{
        1,
        2,
        3,
        4};

    std::size_t required_size = 0;

    // 先验证零长度输入在进入 C++ Core 前得到明确状态。
    const sdk_status empty_status =
        sdk_process(
            nullptr,
            0,
            nullptr,
            0,
            &required_size);
    assert(empty_status == SDK_STATUS_EMPTY_INPUT);

    // 非空输入的第一次调用不提供输出存储，只查询所需容量。
    const sdk_status query_status =
        sdk_process(
            input.data(),
            input.size(),
            nullptr,
            0,
            &required_size);

    assert(
        query_status ==
        SDK_STATUS_BUFFER_TOO_SMALL);
    assert(required_size == input.size());

    // C 调用方拥有并管理实际输出 buffer。
    std::vector<std::uint8_t> output(required_size);

    const sdk_status process_status =
        sdk_process(
            input.data(),
            input.size(),
            output.data(),
            output.size(),
            &required_size);

    // 成功后，只有 required_size 范围内的输出有效。
    assert(process_status == SDK_STATUS_OK);
    assert(
        output ==
        std::vector<std::uint8_t>({2, 3, 4, 5}));
}

### 10. Kotlin/Native wrapper 恢复 Kotlin 语义

Kotlin wrapper 可以：

- 在 `usePinned` 作用域内把 `ByteArray` 借给 C；
- 先查询长度，再创建 Kotlin 拥有的 `ByteArray`；
- 将 status 转换为 sealed result、异常或 nullable；
- 确保 native pointer 不逃逸 pinned 作用域；
- 对 opaque handle 使用 `close/use` 或显式 destroy。

wrapper 不能因为 C++ Core 内部使用 vector/span/variant，就假设这些对象能直接映射成 Kotlin 对象。跨边界的是 C contract，不是 STL 实现。

### 11. 最终语义翻译验收

看到下面的 API，应能直接回答：

```cpp
std::vector<std::uint8_t> read_file();
// 返回自己拥有的数据，调用方接管结果 owner。

void process(std::span<const std::uint8_t> input);
// 调用期间借用只读连续 buffer，不保存、不释放、不修改。

std::optional<User> find_user(std::string_view name);
// 借用只读字符串进行查询；返回拥有的 User，或者正常不存在。

std::variant<Result, Error> execute();
// 返回并拥有 Result 或 Error 中恰好一种状态。
```

不能只回答这些是 STL 模板。答案必须包含 ownership、borrow、mutability、lifetime 和 state contract。

### 12. 最终关键问题

为什么下面的接口适合作为 C++ 内部 API，却绝对不应直接作为 C ABI 暴露？

```cpp
std::variant<
    std::vector<std::uint8_t>,
    Error
>
process(
    std::span<const std::uint8_t> input
);
```

合格答案应覆盖：

1. 输入 span 是同步只读借用，owner 在调用方；
2. 成功 vector 是拥有型动态输出，错误则是 Error alternative；
3. variant 管理 active state，vector 管理分配与析构；
4. 三种 STL 类型都依赖 C++ ABI、对象布局、构造析构、模板实例化、标准库及 compiler/runtime compatibility；
5. C ABI 应翻译为 pointer + length、固定 status、out parameter 或 opaque handle；
6. C++ 异常必须在边界内转换，分配和释放责任必须属于同一明确协议。

### 本阶段边界

本阶段暂不系统展开 list、deque、map、unordered_map、set、allocator、ranges 和 iterator category。它们可以在项目出现真实需求时继续学习。

当前更重要的是形成稳定判断：先从类型读 ownership 和 lifetime，再设计内部 Core；到 ABI 边界时主动把 C++ 类型翻译成简单、稳定、可验证的 C contract。

### 本实验结论

现代 C++ API 的价值不只在语法更短，而在类型可以同时表达 ownership、borrow、mutability、contiguous storage、absence 和 alternative state。

`span<const T>` 适合内部同步借用，`vector<T>` 适合交付拥有型动态结果，`optional<T>` 适合正常缺失，`variant<A, B>` 适合封闭状态集合。组合这些类型可以得到清晰的 C++ Core contract。

跨 C ABI 时必须放弃 STL 对象表示，保留其语义：用 pointer + length 表达借用，用 caller buffer 或 handle 表达所有权，用固定 status/tag 和 out parameter 表达结果。这个翻译层正是后续 Kotlin/Native cinterop 的基础。